<a href="https://colab.research.google.com/github/Andoniadou-Lab/consensus_pituitary_atlas/blob/main/Fig_5/Figure_5_pituitary_atlas_LIANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This notebook shows our workflow for ligand-receptor interaction prediction with LIANA+ after which combine the results using Robust Rank Aggregation (RRA)

Installing packages

In [ ]:
!pip install scanpy
!pip install liana
!pip install --force-reinstall -v "scipy==1.10"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 168.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 90.2 MB/s eta 0:0

In [ ]:
!pip install --force-reinstall -v "scipy==1.12"

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.10'): https://files.pythonhosted.org/packages/99/f1/c00d6be56e1a718a3068079e3ec8ce044d7179345280f6a3f5066068af0d/scipy-1.6.2.tar.gz (from https://pypi.org/simple/scipy/) (requires-python:>=3.7,<3.10)
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.10'): https://files.pythonhosted.org/packages/fe/fd/8704c7b7b34cdac850485e638346025ca57c5a859934b9aa1be5399b33b7/scipy-1.6.3.tar.gz (from https://pypi.org/simple/scipy/) (requires-python:>=3.7,<3.10)
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.10'): https://files.pythonhosted.org/packages/bb/bb/944f559d554df6c9adf037aa9fc982a9706ee0e96c0d5beac701cb158900/scipy-1.7.0.tar.gz (from https://pypi.org/simple/scipy/) (requires-python:>=3.7,<3.10)
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.10'): https://files.pythonhosted.org/packages/47/33/a24aec22b7be7fdb10ec

In [ ]:
#!pip install -i https://test.pypi.org/simple/ "scikit-misc==0.2.0rc1"

# Mounting google drive

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# import liana
import liana as li
import scanpy as sc

In [ ]:
import gc

In [ ]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 11.8 MB/s eta 0:00:00


# Loading wild-type, normal/control datasets

In [ ]:
#load in "/content/drive/MyDrive/pituitary_atlas/Unstructured/pdatas.h5ad"
adata = sc.read_h5ad("/content/drive/MyDrive/pituitary_atlas/Unstructured/wt_adata_merged_post_scvi_2026_02_14.h5ad")
#print shape
print(adata.shape)
#remove those where assignment Blood
adata = adata[adata.obs.new_cell_type != 'Erythrocytes']
gc.collect()

(720485, 32684)


0

In [ ]:
#filter genes in less than 200 cells
sc.pp.filter_genes(adata, min_cells=200)

/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_simple.py:278: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.


In [ ]:
adata.shape

(719113, 25938)

In [ ]:
gc.collect()

706

In [ ]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/pituitary_atlas/source_table/pituitary_atlas.xlsx')
pituitary_atlas = df.copy()

# Ensuring that updated correct gene names are present

In [ ]:
print("Lhb" in adata.var_names)
print("ENSMUSG00000100916.4" in adata.var_names)


True
False


In [ ]:
import gc
gc.collect()

1523

# Generating merged resource used in the analysis

In [ ]:
resource_cellchat = li.rs.select_resource('cellchatdb')
resource_cellphone = li.rs.select_resource('cellphonedb')

In [ ]:
#load /content/v5.0.0_interaction_input.csv
import pandas as pd
interaction_input = pd.read_csv('/content/drive/MyDrive/pituitary_atlas/Unstructured/v5.0.0_interaction_input.csv')
#keep columns protein_name_a and protein_name_b
interaction_input = interaction_input[['protein_name_a', 'protein_name_b']]
#remove where iether is nan
interaction_input = interaction_input.dropna()
#remove from each _HUMAN
interaction_input['protein_name_a'] = interaction_input['protein_name_a'].str.replace('_HUMAN', '')
interaction_input['protein_name_b'] = interaction_input['protein_name_b'].str.replace('_HUMAN', '')
#turn each first letter capitalise
interaction_input['protein_name_a'] = interaction_input['protein_name_a'].str.capitalize()
interaction_input['protein_name_b'] = interaction_input['protein_name_b'].str.capitalize()

#rename cols to ligand and receptor
interaction_input.columns = ['ligand', 'receptor']

In [ ]:
#resource_cellphone if any ligand or receptor is actually two genes, create instead two rows. out of that row, with each gene in its own row.
import pandas as pd

# Split genes in each column
resource_cellchat = resource_cellchat.assign(
    ligand=resource_cellchat['ligand'].str.split('_'),
    receptor=resource_cellchat['receptor'].str.split('_')
).explode('ligand').explode('receptor').reset_index(drop=True)
#remove duplicate rows
resource_cellchat = resource_cellchat.drop_duplicates()
#turn all entries to first capital only
resource_cellchat['ligand'] = resource_cellchat['ligand'].str.capitalize()
resource_cellchat['receptor'] = resource_cellchat['receptor'].str.capitalize()

#repeat for resource_cellphone
resource_cellphone = resource_cellphone.assign(
    ligand=resource_cellphone['ligand'].str.split('_'),
    receptor=resource_cellphone['receptor'].str.split('_')
).explode('ligand').explode('receptor').reset_index(drop=True)
#remove duplicate rows
resource_cellphone = resource_cellphone.drop_duplicates()
#turn all entries to first capital only
resource_cellphone['ligand'] = resource_cellphone['ligand'].str.capitalize()
resource_cellphone['receptor'] = resource_cellphone['receptor'].str.capitalize()

mouseconsensus = pd.concat([resource_cellchat, resource_cellphone, interaction_input])
mouseconsensus = mouseconsensus.drop_duplicates()
#reset
mouseconsensus = mouseconsensus.reset_index(drop=True)
print(mouseconsensus.shape)
mouseconsensus.head(5)


(3268, 2)


,ligand,receptor
0,Jag2,Notch1
1,Dll1,Notch1
2,Igf1,Igf1r
3,Jag1,Notch1
4,Wnt5a,Fzd2


In [ ]:
#save mouseconsensus to Unstructured
mouseconsensus.to_csv('/content/drive/MyDrive/pituitary_atlas/Unstructured/mouseconsensus.csv')

In [ ]:
# import all individual methods
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean, rank_aggregate

In [ ]:
adata.obs.SRA_ID.nunique()

169

In [ ]:
adata.obs.new_cell_type.value_counts()

,count
new_cell_type,
Lactotrophs,248405
Somatotrophs,239572
Stem_cells,56688
Melanotrophs,51338
Corticotrophs,36420
Gonadotrophs,27565
Thyrotrophs,15245
Mesenchymal_cells,13141
Immune_cells,12000


In [ ]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/pituitary_atlas/source_table/pituitary_atlas.xlsx')
#make sure Sorted is = 0 and Normal is 1 and Modality is sc, sn or multi_rna
df = df[(df['Sorted'] == 0) & (df['Normal'] == 1) & (df['Modality'].isin(['sc','sn','multi_rna']))]
#keep those that are in adata.obs.SRA_ID
df = df[df['SRA_ID'].isin(adata.obs.SRA_ID.unique())]
df

,GEO,SRA_ID,Name,Conditions,Normal,Background,Sorted,Author,DOI,Age,...,filtering_junk,median_cellassign_prob,most_recent_workflow,n_cells,processed_atac,passed_qc_tcc,tcc_pseudobulked,Core,Notes 1,Notes 2
0,GSM4594333,SRX8489818,M-Pituitary,normal,1,C57BL/6,0,Ruf-Zamojski et al. (2021),10.1038/s41467-021-22859-w,10-12 weeks,...,1.0,1.0,v_0.01,1278,0.0,1.0,v_0.01,1,NaN,NaN
1,GSM4594334,SRX8489819,FM_Pit_1-1,normal,1,C57BL/6,0,Ruf-Zamojski et al. (2021),10.1038/s41467-021-22859-w,10-12 weeks,...,1.0,1.0,v_0.01,8236,0.0,1.0,v_0.01,1,NaN,NaN
2,GSM4594335,SRX8489820,FM_Pit_1-2,normal,1,C57BL/6,0,Ruf-Zamojski et al. (2021),10.1038/s41467-021-22859-w,10-12 weeks,...,1.0,1.0,v_0.01,7092,0.0,1.0,v_0.01,1,NaN,NaN
3,GSM4594336,SRX8489821,MM_Pit_4-1,normal,1,C57BL/6,0,Ruf-Zamojski et al. (2021),10.1038/s41467-021-22859-w,10-12 weeks,...,1.0,1.0,v_0.01,1755,0.0,1.0,v_0.01,1,NaN,NaN
4,GSM4594337,SRX8489822,MM_Pit_1v2,normal,1,C57BL/6,0,Ruf-Zamojski et al. (2021),10.1038/s41467-021-22859-w,10-12 weeks,...,1.0,1.0,v_0.01,416,0.0,1.0,v_0.01,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,GSM7663102,SRX21168596,"Pituitary, aged, sample2, scRNA-seq",normal,1,C57BL/6,0,Wei et al. (2025),NaN,24 months,...,1.0,1.0,v_0.01,6507,NaN,NaN,NaN,1,missing sex,NaN
265,MMP3-CONT1,MMP3-CONT1,Pituitary_ctrl,normal,1,C57BL/6,0,Kover et al. (2025),NaN,4 days,...,1.0,NaN,v_0.01,2451,NaN,NaN,NaN,2,NaN,NaN
279,GSM9302174,SRX31166360,"pituitary, ND_1",normal,1,Unclear,0,Guo et al. (2025),NaN,16 weeks,...,1.0,1.0,v_0.01,7298,NaN,NaN,NaN,1,NaN,NaN
280,GSM9302175,SRX31166361,"pituitary, ND_2",normal,1,Unclear,0,Guo et al. (2025),NaN,16 weeks,...,1.0,1.0,v_0.01,7530,NaN,NaN,NaN,1,in previous version Normal was switched as 0 w...,NaN


# Removing non-anterior pituitary cell type (unlikely to interact in a paracrine way due to spatial segregation)

In [ ]:
#remove Pituicytes
adata = adata[adata.obs.new_cell_type != 'Pituicytes']

In [ ]:
#keep only whole pits, to not bias the rank aggregation step to sorted cell types!
adata = adata[adata.obs.SRA_ID.isin(df['SRA_ID'].unique())]

In [ ]:
adata

View of AnnData object with n_obs × n_vars = 682983 × 25938
    obs: '0', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_malat', 'log1p_total_counts_malat', 'pct_counts_malat', 'doublet_score', 'predicted_doublet', 'size_factor', 'assignments', 'original_barcode', 'new_cell_type', 'SRA_ID', 'GEO', 'assay', 'orig_index', 'barcode', 'predicted_cell_type', 'predicted_cell_type_proba', 'proba_Corticotrophs', 'proba_Endothelial_cells', 'proba_Erythrocytes', 'proba_Gonadotrophs', 'proba_Immune_cells', 'proba_Lactotrophs', 'proba_Melanotrophs', 'proba_Mesenchymal_cells', 'proba_Pituicytes', 'proba_Somatotrophs', 'proba_Stem_cells', 'proba_Thyrotrophs', 'cell_type_final', 'pct_counts_kept', 'init_predicted_doublet_epitome', 'thresholded_doublet_epitome', 'doublet_score_epitome', 'Name', 'Condi

In [ ]:
adata.X[:10,:10].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

# Main loop running LIANA+ for each dataset separately

In [ ]:
list_of_dfs = []
for sra_id in adata.obs.SRA_ID.unique():
  subset = adata[adata.obs.SRA_ID == sra_id].copy()
  sc.pp.normalize_total(subset, target_sum=1e4)
  sc.pp.log1p(subset)
  li.mt.rank_aggregate(subset,
                     groupby='new_cell_type',
                     resource=mouseconsensus,
                     expr_prop=0.1,
                     use_raw=False,
                     n_perms=1000,
                     verbose=True)
  list_of_dfs.append(subset.uns['liana_res'])



/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


Using provided `resource`.


/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Generating ligand-receptor stats for 8073 samples and 888 features


/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:293: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:296: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.


...

Using provided `resource`.
Generating ligand-receptor stats for 1264 samples and 808 features
Assuming that counts were `natural` log-normalized!


/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:293: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
/usr/local/lib/python3.12/dist-packages/liana/method/sc/_liana_pipe.py:296: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.


Running CellPhoneDB


100%|██████████| 1000/1000 [00:01<00:00, 798.12it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


In [ ]:
len(list_of_dfs)

160

In [ ]:
lists_of_np_arrays = []
for df in list_of_dfs:
  #concat ligand_complex receptor_complex
  df["concat_entry"] = df["ligand_complex"] + "___" + df["receptor_complex"] + "___" + df["target"] + "___" + df["source"]
  #order by specificity_rank
  df = df.sort_values(by=['specificity_rank'],ascending=True)
  lists_of_np_arrays.append(df["concat_entry"].to_numpy())


lists_of_np_arrays_magnitude = []
for df in list_of_dfs:
  #concat ligand_complex receptor_complex
  df["concat_entry"] = df["ligand_complex"] + "___" + df["receptor_complex"] + "___" + df["target"] + "___" + df["source"]
  #order by specificity_rank
  df = df.sort_values(by=['magnitude_rank'],ascending=True)
  lists_of_np_arrays_magnitude.append(df["concat_entry"].to_numpy())



# Defining the robust rank aggregation function

Within this, we also remove interactions that are not spatially plausible, including intermediate lobe melanotrophs signalling to anterior cell types.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, beta

#import for FDR
from statsmodels.stats.multitest import multipletests

def robust_rank_aggregation(lists):
    """
    Implements the Robust Rank Aggregation method from Kolde et al. (2012)

    Parameters:
    -----------
    lists : list of lists/arrays
        Each list contains gene names in order of rank

    Returns:
    --------
    pd.DataFrame with aggregated scores and ranks
    """
    # Convert lists to normalized ranks
    all_genes = list(set([gene for sublist in lists for gene in sublist]))
    n_genes = len(all_genes)
    gene_to_idx = {gene: idx for idx, gene in enumerate(all_genes)}

    # Create rank matrix (rows=genes, cols=lists)
    rank_matrix = np.ones((n_genes, len(lists)))

    # Fill rank matrix with normalized ranks
    for list_idx, gene_list in enumerate(lists):
        # Convert to ranks, handling ties with average
        list_length = len(gene_list)
        ranks = rankdata(range(list_length), method='average') / list_length

        # Fill matrix
        for rank_idx, gene in enumerate(gene_list):
            rank_matrix[gene_to_idx[gene], list_idx] = ranks[rank_idx]

    # Calculate rho scores for each gene
    rho_scores = np.zeros(n_genes)
    for i in range(n_genes):
        # Sort ranks for this gene
        gene_ranks = np.sort(rank_matrix[i])

        # Calculate beta distribution parameters for each rank
        k_values = np.arange(1, len(lists) + 1)
        dist_a = k_values
        dist_b = len(lists) - k_values + 1

        # Calculate p-values using beta distribution
        p_values = np.array([
            beta.cdf(r, a, b)
            for r, a, b in zip(gene_ranks, dist_a, dist_b)
        ])

        # Take minimum p-value
        rho_scores[i] = np.min(p_values)

    # Bonferroni correction
    print(len(lists))
    print(rho_scores.shape)
    #corrected_scores = np.minimum(rho_scores * len(rho_scores), 1.0)


    # Create result DataFrame
    result = pd.DataFrame({
        'gene': all_genes,
        'rho_score': rho_scores,
        'corrected_score': rho_scores,
    })

    #remove those genes where it ends with
    #Melanotrophs___Gonadotrophs
    #Melanotrophs___Lactotrophs
    #Melanotrophs___Somatotrophs
    #Melanotrophs___Thyrotrophs
    #Melanotrophs___Corticotrophs - because these cells don't directly interface with each other.

    result = result[~result['gene'].str.endswith('Melanotrophs___Gonadotrophs')]
    result = result[~result['gene'].str.endswith('Melanotrophs___Lactotrophs')]
    result = result[~result['gene'].str.endswith('Melanotrophs___Somatotrophs')]
    result = result[~result['gene'].str.endswith('Melanotrophs___Thyrotrophs')]
    result = result[~result['gene'].str.endswith('Melanotrophs___Corticotrophs')]

    result = result[~result['gene'].str.endswith('Gonadotrophs___Melanotrophs')]
    result = result[~result['gene'].str.endswith('Lactotrophs___Melanotrophs')]
    result = result[~result['gene'].str.endswith('Somatotrophs___Melanotrophs')]
    result = result[~result['gene'].str.endswith('Thyrotrophs___Melanotrophs')]
    result = result[~result['gene'].str.endswith('Corticotrophs___Melanotrophs')]

    # FDR
    _, corrected_scores, _, _ = multipletests(result["rho_score"], method='fdr_bh')

    result['corrected_score'] = corrected_scores

    result['rank'] = result['corrected_score'].rank(method='dense')

    return result.sort_values('corrected_score')

# Example usage
def test_rra():
    # Example lists of different lengths
    lists = [
        ["A", "B", "C", "D", "E"],
        ["B", "A", "C", "E", "D"],
        ["B", "C", "A", "D"],
        ["C", "B", "A", "E"],
    ]

    result = robust_rank_aggregation(lists)
    print("\nRRA Results:")
    print(result)
    return result

test_rra()

4
(5,)

RRA Results:
  gene  rho_score  corrected_score  rank
0    B   0.062500         0.312500   1.0
1    C   0.129600         0.324000   2.0
4    A   0.316406         0.527344   3.0
2    D   0.998400         0.998400   4.0
3    E   0.998400         0.998400   4.0


,gene,rho_score,corrected_score,rank
0,B,0.062500,0.312500,1.0
1,C,0.129600,0.324000,2.0
4,A,0.316406,0.527344,3.0
2,D,0.998400,0.998400,4.0
3,E,0.998400,0.998400,4.0


# running the aggregation for magnitude and specificity scores separately

In [ ]:
rra_magnitude = robust_rank_aggregation(lists_of_np_arrays_magnitude)

160
(84820,)


In [ ]:
rra_spec = robust_rank_aggregation(lists_of_np_arrays)

160
(84820,)


In [ ]:
#merge rra_spec and rra_magnitude on gene
merged = pd.merge(rra_spec, rra_magnitude, on='gene')

# saving the merged results

In [ ]:
#save as parquet called liana_consensus
merged.to_parquet('/content/drive/MyDrive/pituitary_atlas/Unstructured/liana_consensus_2026_02_16.parquet')
merged.to_parquet('/content/liana_consensus_2026_02_16.parquet')

In [ ]:
merged.head(100)

,gene,rho_score_x,corrected_score_x,rank_x,rho_score_y,corrected_score_y,rank_y
0,Nrg3___Erbb4___Stem_cells___Melanotrophs,8.029041e-181,6.163814e-176,1.0,8.573515e-236,1.096967e-231,3.0
1,Lama2___Itga1___Endothelial_cells___Mesenchyma...,3.151156e-160,1.209555e-155,2.0,5.283625e-49,1.300060e-46,307.0
2,Lama2___Sv2b___Melanotrophs___Mesenchymal_cells,2.212175e-137,5.660882e-133,3.0,1.584127e-54,4.554749e-52,262.0
3,Vegfd___Kdr___Endothelial_cells___Gonadotrophs,5.991563e-135,1.149916e-130,4.0,5.357500e-32,8.393672e-30,482.0
4,Col1a1___Sdc4___Stem_cells___Mesenchymal_cells,4.249624e-126,6.524788e-122,5.0,2.441652e-41,5.079761e-39,364.0
...,...,...,...,...,...,...,...
95,Col3a1___Itgb1___Endothelial_cells___Mesenchym...,1.190395e-68,9.519320e-66,95.0,2.821338e-40,5.655124e-38,378.0
96,Tgfb2___Tgfbr3___Gonadotrophs___Stem_cells,1.236278e-68,9.784314e-66,96.0,7.289266e-38,1.338731e-35,411.0
97,Col4a1___Itga9___Immune_cells___Endothelial_cells,2.348983e-68,1.821506e-65,97.0,1.842964e-35,3.151058e-33,441.0
98,Lama2___Itgav___Stem_cells___Mesenchymal_cells,2.347066e-68,1.821506e-65,97.0,1.374670e-34,2.284243e-32,454.0


In [ ]:
#find where gene ends with Stem_cells
merged[merged['gene'].str.endswith('Stem_cells')].head(50)

,gene,rho_score_x,corrected_score_x,rank_x,rho_score_y,corrected_score_y,rank_y
30,Fgf1___Fgfr1___Stem_cells___Stem_cells,2.511826e-90,6.220334e-87,31.0,4.647252e-06,2.167466e-04,1615.0
65,Slit2___Robo2___Lactotrophs___Stem_cells,2.255658e-74,2.623707e-71,65.0,5.321782e-160,1.857036e-156,18.0
72,Lama5___Sv2b___Melanotrophs___Stem_cells,4.107516e-73,4.319588e-70,72.0,1.845086e-03,6.741810e-02,2056.0
76,Slit2___Robo1___Lactotrophs___Stem_cells,3.256626e-72,3.246856e-69,76.0,2.474538e-155,7.915326e-152,20.0
84,Lama5___Itga6___Stem_cells___Stem_cells,1.990635e-71,1.797871e-68,84.0,1.482209e-03,5.470564e-02,2035.0
85,Sema6d___Kdr___Endothelial_cells___Stem_cells,8.310173e-71,7.418182e-68,85.0,1.537401e-16,1.304141e-14,893.0
90,Frem2___Itgb1___Stem_cells___Stem_cells,1.737242e-69,1.465564e-66,90.0,4.914914e-09,2.845498e-07,1306.0
96,Tgfb2___Tgfbr3___Gonadotrophs___Stem_cells,1.236278e-68,9.784314e-66,96.0,7.289266e-38,1.338731e-35,411.0
105,Lama5___Itgb8___Stem_cells___Stem_cells,1.803154e-66,1.305909e-63,104.0,9.856629e-01,1.000000e+00,2385.0
106,Cntn1___Nrcam___Stem_cells___Stem_cells,2.323930e-66,1.667344e-63,105.0,4.348870e-106,3.589875e-103,88.0


In [ ]:
pip freeze

absl-py==1.4.0
accelerate==1.12.0
aiofiles==25.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
altair==5.5.0
anndata==0.12.10
annotated-types==0.7.0
anyio==4.12.1
apsw==3.51.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array-api-compat==1.13.0
array_record==0.8.3
arrow==1.4.0
attrs==25.4.0
audioread==3.1.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bleach==6.3.0
blinker==1.4
blis==1.3.3
catalogue==2.0.10
certifi==2026.1.4
cffi==2.0.0
charset-normalizer==3.4.4
chex==0.1.90
click==8.3.1
cloudpathlib==0.23.0
cloudpickle==3.1.2
confection==0.1.5
contourpy==1.3.3
cryptography==3.4.8
cycler==0.12.1
cymem==2.0.13
datasets==4.5.0
dbus-python==1.2.18
debugpy==1.8.15
decorator==5.2.1
defusedxml==0.7.1
dill==0.4.0
distrax==0.1.5
distro==1.9.0
dm-tree==0.1.9
docrep==0.3.2
docstring_parser==0.17.0
donfig==0.8.1.post1
einops==0.8.2
en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/